# Comparação didática de métodos de HPO

Este notebook compara **Grid Search**, **Random Search** e **TPE** sob o mesmo orçamento de 60 avaliações. A função objetivo simula uma perda de validação: ela é rápida, determinística e permite observar o comportamento das estratégias sem baixar dados ou treinar uma rede.

> **Escopo.** Esta é uma ilustração dos conceitos do Seminário 3, e não uma reprodução dos benchmarks de Bergstra et al. (2011) nem do Auto-PyTorch. Em especial, TPE não é BOHB, e este notebook não implementa múltiplas fidelidades, portfólio meta-aprendido ou ensembles.

## Dependências

Se necessário, execute em uma célula separada: `%pip install numpy pandas matplotlib optuna`.

In [ ]:
%pip install numpy pandas matplotlib optuna

In [ ]:
from itertools import product

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import optuna
    import pandas as pd
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        'Instale as dependências com: %pip install numpy pandas matplotlib optuna'
    ) from error

optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
TRIALS = 60

learning_rates = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
dropouts = [0.0, 0.2, 0.6]
widths = [32, 64, 128, 256]

assert len(learning_rates) * len(dropouts) * len(widths) == TRIALS
print(f'Orçamento: {TRIALS} avaliações por método (semente {SEED}).')

## Função objetivo simulada

O mínimo está próximo de `learning_rate=1e-3`, `dropout=0.2` e `width=128`. Um ruído pequeno e determinístico, calculado a partir da própria configuração, torna a superfície menos idealizada sem prejudicar a comparação: os três métodos recebem exatamente a mesma perda para uma mesma configuração. Em um experimento real, esta função seria substituída pelo treino e validação de um modelo.

In [ ]:
def objective(learning_rate, dropout, width):
    """Perda de validação simulada; menor é melhor."""
    log_learning_rate = np.log10(learning_rate)
    deterministic_noise = 0.002 * np.sin(
        17 * log_learning_rate + 11 * dropout + width / 32
    )
    return (
        (log_learning_rate + 3) ** 2
        + 2 * (dropout - 0.2) ** 2
        + ((width - 128) / 128) ** 2
        + deterministic_noise
    )

objective(1e-3, 0.2, 128)

In [ ]:
def grid_search():
    records = []
    for trial, (learning_rate, dropout, width) in enumerate(
        product(learning_rates, dropouts, widths), start=1
    ):
        records.append({
            'method': 'Grid Search',
            'trial': trial,
            'learning_rate': learning_rate,
            'dropout': dropout,
            'width': width,
            'loss': objective(learning_rate, dropout, width),
        })
    return pd.DataFrame(records)


def optuna_search(name, sampler):
    records = []

    def evaluate(trial):
        learning_rate = trial.suggest_categorical('learning_rate', learning_rates)
        dropout = trial.suggest_categorical('dropout', dropouts)
        width = trial.suggest_categorical('width', widths)
        loss = objective(learning_rate, dropout, width)
        records.append({
            'method': name,
            'trial': trial.number + 1,
            'learning_rate': learning_rate,
            'dropout': dropout,
            'width': width,
            'loss': loss,
        })
        return loss

    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(evaluate, n_trials=TRIALS, show_progress_bar=False)
    return pd.DataFrame(records)


grid_results = grid_search()
random_results = optuna_search('Random Search', optuna.samplers.RandomSampler(seed=SEED))
tpe_results = optuna_search('TPE', optuna.samplers.TPESampler(seed=SEED))
results = pd.concat([grid_results, random_results, tpe_results], ignore_index=True)
results.head()

## Resultados

A grade cobre todas as 60 combinações discretas e, por isso, encontra o mínimo desse espaço. Random Search e TPE fazem 60 tentativas no mesmo espaço, podendo repetir configurações. Em espaços grandes ou contínuos, a grade se torna rapidamente inviável; é aí que métodos amostrais e sequenciais se tornam especialmente úteis.

In [ ]:
best_results = (
    results.loc[results.groupby('method')['loss'].idxmin()]
    .sort_values('loss')
    .reset_index(drop=True)
)
best_results[['method', 'loss', 'learning_rate', 'dropout', 'width']]

In [ ]:
plot_data = results.copy()
plot_data['best_loss_so_far'] = plot_data.groupby('method')['loss'].cummin()

fig, ax = plt.subplots(figsize=(9, 5))
for method, history in plot_data.groupby('method', sort=False):
    ax.plot(history['trial'], history['best_loss_so_far'], label=method, linewidth=2)

ax.set_title('Convergência: melhor perda observada até cada avaliação')
ax.set_xlabel('Avaliação')
ax.set_ylabel('Melhor perda até o momento (menor é melhor)')
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Leitura do experimento

- **Grid Search** é uma referência útil em espaços pequenos, mas seu custo cresce exponencialmente com o número de valores e hiperparâmetros.
- **Random Search** explora combinações sem usar o histórico das tentativas; é um baseline simples e forte.
- **TPE** usa o histórico para priorizar regiões promissoras. Uma única execução não prova superioridade universal: o desempenho depende da função objetivo, do espaço, do orçamento e da semente.
- **BOHB/Auto-PyTorch** acrescentam alocação de orçamento por múltiplas fidelidades e outros componentes; portanto, não são equivalentes a este estudo com Optuna.

## Mapa das avaliações realizadas

Cada ponto é uma configuração realmente avaliada. A cor representa a perda e o tamanho representa a largura da camada. Em problemas reais, esse tipo de gráfico ajuda a observar se a busca está se concentrando em regiões de menor perda.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (method, history) in zip(axes, results.groupby('method', sort=False)):
    points = ax.scatter(
        np.log10(history['learning_rate']), history['dropout'],
        c=history['loss'], s=history['width'] / 2, cmap='viridis', alpha=0.8
    )
    ax.set_title(method)
    ax.set_xlabel('log10(learning_rate)')
    ax.grid(alpha=0.25)
axes[0].set_ylabel('dropout')
fig.colorbar(points, ax=axes, label='Perda simulada')
fig.suptitle('Mapa das configurações avaliadas (tamanho = width)', y=1.03)
plt.show()

## Successive Halving simulado

A seguir, 27 configurações recebem um orçamento baixo; as 9 melhores avançam para orçamento intermediário e as 3 melhores recebem orçamento alto. A métrica em baixa fidelidade contém uma perturbação maior, ilustrando que avaliações iniciais são baratas, mas imperfeitas. Isto mostra a ideia de Hyperband/BOHB, mas **não** implementa BOHB: não há modelo bayesiano nem amostragem TPE entre estágios.

In [ ]:
def loss_at_budget(config, budget):
    learning_rate, dropout, width = config
    base_loss = objective(learning_rate, dropout, width)
    low_fidelity_variation = 0.18 / budget * np.sin(
        23 * np.log10(learning_rate) + 7 * dropout + width / 16
    )
    return base_loss + low_fidelity_variation

all_configs = list(product(learning_rates, dropouts, widths))
rng = np.random.default_rng(SEED)
survivors = [all_configs[index] for index in rng.choice(len(all_configs), size=27, replace=False)]
stages = [(1, 9), (3, 3), (9, 1)]
halving_records = []

for stage, (budget, promoted_count) in enumerate(stages, start=1):
    evaluated = []
    for config in survivors:
        loss = loss_at_budget(config, budget)
        evaluated.append((loss, config))
        halving_records.append({
            'stage': stage, 'budget': budget, 'loss': loss,
            'learning_rate': config[0], 'dropout': config[1], 'width': config[2],
        })
    survivors = [config for _, config in sorted(evaluated)[:promoted_count]]

halving_results = pd.DataFrame(halving_records)
stage_summary = halving_results.groupby(['stage', 'budget'], as_index=False).agg(
    configurations=('loss', 'size'), best_loss=('loss', 'min')
)
used_budget = int((halving_results['budget']).sum())
full_budget = 27 * 9
print(f'Orçamento usado: {used_budget} unidades; treino completo de 27 configs: {full_budget}.')
stage_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(stage_summary['stage'].astype(str), stage_summary['configurations'], color='#4c78a8')
ax.set_title('Configurações avaliadas por estágio')
ax.set_xlabel('Estágio (orçamento: 1, 3 e 9)')
ax.set_ylabel('Número de configurações')
for stage, count in zip(stage_summary['stage'], stage_summary['configurations']):
    ax.text(stage - 1, count + 0.5, str(count), ha='center')
ax.set_ylim(0, 30)
plt.show()

finalist = halving_results.query('stage == 3').nsmallest(1, 'loss')
finalist[['learning_rate', 'dropout', 'width', 'loss']]